# **Return of the Schema** for  a *General User Defined Knowledge Graph* 

This Notebook will guide you in using the provided KG-SaF-JDeX Workflow Functionalities to generate a new Schema and Data Dataset from your custom Knowledge Graph. Please follow the README guide before going forward. This guide provides an example dataset in the INPUT folder to test the functionalities. All the produces files will be created in the OUTPUT folder.  

This notebook expectes the following inputs:

- Any Knowledge Graph with both Schema and Data in a unique file (any format supported by the ROBOT Utility, the file will be converted to an intermediate OWL File)
- The KG need to contain ABox assertions (object property assertions) in order to safely run the machine learning splitting and necessary checks
- This module is specifically designed for generating dataset from Knowledge graphs that contain rich schema and large scale ABox (object property assertions)

And applies the following procedures following the KG-SaF-JDeX workflow:

#### Dataset Creation Steps

- Materialization and Realization
- Filtering of ABox Individuals and Object Properties
- Object Propety Assertion splitting using Coverage Criterion in Training, Test and Validatin Split
- Inversion Leakage check and filtering
- Class Assertions subset computation
- Schema Modularization based on ABox Signature
- Schema Decomposition in TBox and RBox (with subsequent division in Schema and Taxonomy)
- Fulle cleaned Ontology and Knowledge Graph Reconstruction


#### Dataset Machine Learning Post Processing Steps

- Conversion and serialization of object property assertion to TSV format
- Conversion and serialization of schema axioms to JSON format

In [ ]:
from rdflib import Graph, RDF, OWL, BNode
from rdflib.term import URIRef
from pathlib import Path
import sys
import subprocess
import json
sys.path.append(str(Path.cwd().parent))
from kgsaf_jdex.utils.conventions.builtins import BUILTIN_URIS
from kgsaf_jdex.utils.modularization import SignatureModularizer, SchemaDecomposer 
from pykeen.triples import TriplesFactory
from pykeen.triples.splitting import CoverageSplitter
from kgsaf_jdex.utils.conversion import OWLConverter, TSVConverter, IDMapper
from pykeen.triples.leakage import unleak
from kgsaf_jdex.utils.reason import ReasonerUtility

## Base Parameters 

In [ ]:
# Do you want to run reasoning services on your dataset? 
REASONER = True

# INPUT Graph file Location
KG_FILE = Path().cwd().absolute() / "input" / "kg_consistent.owl"

# OUTPUT Dataset Location
OUTPUT_PATH = Path().cwd().absolute() / "output"

# Name of the GRAPH (Will be the datset folder NAME)
DATASET_NAME = "kg_consistent"
DATASET_NAME = DATASET_NAME + "_reasoned" if REASONER else DATASET_NAME + "_base"
DATASET_PATH = OUTPUT_PATH / f"{DATASET_NAME}"

# Location of the ROBOT Jar
ROBOT_JAR = "/home/navis/robot/robot.jar"
REPORT_PATH = DATASET_PATH / "report.owl"
OUTPUT_FILE = DATASET_PATH / "intermediate_kg.owl"

print(f"Using resoner? \t\t{REASONER}")
print(f"Input Graph Location \t{KG_FILE}")
print(f"Output Dataset Folder \t{OUTPUT_PATH}")
print(f"Dataset Name \t\t{DATASET_NAME}")
print(f"Robot JAR \t\t{ROBOT_JAR}")
print(f"Output Dataset Path \t{DATASET_PATH}")
print(f"Report Path: \t\t{REPORT_PATH}")

reasoner = ReasonerUtility(ROBOT_JAR)

In [ ]:
print(f"Creating base datset Folders...")

(DATASET_PATH / "abox" / "splits").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "mappings").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "tbox").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "rbox").mkdir(parents=True, exist_ok=True)

print(f"Done!")

#### Conversion of data to OWL Format

In [ ]:
reasoner.convert_owl(KG_FILE, OUTPUT_FILE)

## (OPTIONAL) Reasoning Utilities: Removal of Unsatisfiable Classes

In [ ]:
if REASONER:
    reasoner.filter_unsatisfiable(OUTPUT_FILE, OUTPUT_FILE)

## (OPTIONAL) Reasoning Utilities

This block will run reasoning services for CONSISTENCY CHECK, MATERIALIZATION and REALIZATION

In [ ]:
if REASONER:
    properties = [
        "SubClass",
        "EquivalentClass",
        "EquivalentObjectProperty",
        "InverseObjectProperties",
        "ObjectPropertyCharacteristic",
        "SubObjectProperty",
        "ObjectPropertyRange",
        "ObjectPropertyDomain",
        "ClassAssertion"
    ]

    debug_path = DATASET_PATH / "debug.owl"
    reasoner.reason(properties, OUTPUT_FILE, OUTPUT_FILE, debug_path)

# ABOX Filtering: NamedIndividuals and ObjectProperties

Removal of NamedIndividuals also defined as Classes and ObjectProperties also defined as DatatypeProperties

In [ ]:
print("Parsing and Loading Target Knowledge Graph...")
kg = Graph()
kg.parse(OUTPUT_FILE)
print("Done!")

#### Triples

In [ ]:
print("Filtering <NamedIndividual, ObjectPropety, NamedIndividual> Triples")

triples_graph = Graph()
for s,p,o in kg:
    if (s, RDF.type, OWL.NamedIndividual) in kg and (p, RDF.type, OWL.ObjectProperty) in kg and (o, RDF.type, OWL.NamedIndividual) in kg:
        triples_graph.add((s,p,o))

print(f"Initial Dataset: {len(triples_graph)} triples found (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual) Found")

#### Predicates

In [ ]:
predicates = set()

print("Analyzing Object Properties")

for prop in triples_graph.predicates(None, None):
    if (prop, RDF.type, OWL.DatatypeProperty) not in kg:
        predicates.add(prop)
    else:
        print(f"Predicate {prop} removed from dataset. Defined as DatatypeProperty")

print(f"Initial Dataset: {len(predicates)} predicates (OWL.ObjectProperty) Found")


#### Individuals

In [ ]:
individuals = set()

print("Analyzing SUBJECTS")
for ind in triples_graph.subjects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")

print("Analyzing OBJECTS")
for ind in triples_graph.objects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")

print(f"Dataset will contain a total of {len(individuals)} individuals (OWL.NamedIndividual)")

#### Filtering

In [ ]:
print(f"Removing Triples...")

for s,p,o in triples_graph:
    if (s not in individuals) or (o not in individuals):
        triples_graph.remove((s,p,o))

print(f"Dataset will contain a total of {len(triples_graph)} triples (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual)")

print("Serializing Intermediate File...")
triples_graph.serialize(DATASET_PATH / "obj_prop_assertion.nt", format="nt", robot_jar=ROBOT_JAR)

with open(DATASET_PATH / "obj_prop_assertion.tsv", "w") as f:
    for s, p, o in triples_graph:
        f.write(f"{s}\t{p}\t{o}\n")
print("Done")

## Dataset Splitting: Covarage Splitting in Train Validation and Test, Inversion Leakage Filtering

Modify this cell as need following PyKEEN guidelines to change the Splitting Criterion. Reporting from PyKEEN Docs:

```text
PyKEEN currently only supports the creation of transductive splits. 
One reason for this is that in the inductive setting, there are several different ways to create inductive splits and the literature uses different ones, cf. https://arxiv.org/abs/2107.04894. 
You can still use PyKEEN for inductive link prediction with existing data sets, or with new inductive data sets that you create. 
For a general discussion of inductive link prediction, see Inductive Link Prediction.```

In [ ]:
triples = TriplesFactory.from_path(DATASET_PATH / "obj_prop_assertion.tsv")

entity_mappings = {v:k for k,v in triples.entity_id_to_label.items()}
relation_mappings = {v:k for k,v in triples.relation_id_to_label.items()}

train, valid, test = triples.split(
    ratios=[0.85, 0.05, 0.1],
    random_state=42,
    method=CoverageSplitter(),      
)

train_clean = TriplesFactory.from_labeled_triples(
    triples=train.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

valid_clean = TriplesFactory.from_labeled_triples(
    triples=valid.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

test_clean = TriplesFactory.from_labeled_triples(
    triples=test.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

train_unleak, valid_unleak, test_unleak = unleak(
    train_clean,
    *[valid_clean, test_clean],
    n=None,
    minimum_frequency=0.97
    )


In [ ]:
print("Serializing NT Splits and Full Dataset...")

targets = [
    (DATASET_PATH / "abox/splits/train", train_unleak.triples),
    (DATASET_PATH / "abox/splits/valid", valid_unleak.triples),
    (DATASET_PATH / "abox/splits/test", test_unleak.triples)
]

for path, split in targets:
    out_graph = Graph()
    for triple in split:
        s = URIRef(triple[0])
        p = URIRef(triple[1])
        o = URIRef(triple[2])
        out_graph.add((URIRef(s), URIRef(p), URIRef(o)))

    print(f"\tSerializing {path}")
    out_graph.serialize(path.with_suffix(".nt"), format="nt")

print(f"\tSerializing  Full Dataset")
!cat {DATASET_PATH}/abox/splits/*.nt > {DATASET_PATH}/abox/obj_prop_assertions.nt
print("Done")

## ABox Serialization: NamedIndividuals and ClassAssertions

In [ ]:
print("Serializing NamedIndividuals...")
out_graph = Graph()

for ind in individuals:
    out_graph.add((ind, RDF.type, OWL.NamedIndividual))

reasoner.serialize(out_graph, DATASET_PATH / "abox" / "individuals")
del out_graph
print("Done!")

In [ ]:
print("Serializing ClassAssertions...")
class_assertions_graph = Graph()

for ind in individuals:
    for ca in set(kg.objects(ind, RDF.type)) - BUILTIN_URIS:
        if (ca, RDF.type, OWL.Class) in kg:
            class_assertions_graph.add((ind, RDF.type, ca))
        else:
            print(f"Not a Class {ca}")

reasoner.serialize(class_assertions_graph, DATASET_PATH / "abox" / "class_assertions")
print("Done!")

# Schema Subset: Signature Based Modularization

In [ ]:
seed_obj_props = predicates
print("Seed Object Properties", len(seed_obj_props))

seed_classes = set(class_assertions_graph.objects(None, RDF.type))
print("Seed Classes", len(seed_classes))

modularizer = SignatureModularizer(kg, seed_classes | seed_obj_props)
out_graph = modularizer.modularize(verbose=False)

reasoner.serialize(out_graph, DATASET_PATH / "ontology")

## Schema Decomposition: TBox and RBox

In [ ]:
onto_graph = Graph()
onto_graph.parse(DATASET_PATH / "ontology.owl")

decomposer = SchemaDecomposer(onto_graph)
rbox_graph, taxonomy_graph, schema_graph = decomposer.decompose(verbose=False)

reasoner.serialize(rbox_graph, DATASET_PATH / "rbox" / "roles")
reasoner.serialize(taxonomy_graph, DATASET_PATH / "tbox" / "taxonomy")
reasoner.serialize(schema_graph, DATASET_PATH / "tbox" / "schema")

# Ontology and Knowledge Graph Recomposition

In [ ]:
(DATASET_PATH / "intermediate_kg.owl").unlink()
(DATASET_PATH / "obj_prop_assertion.nt").unlink()
(DATASET_PATH / "obj_prop_assertion.tsv").unlink()

In [ ]:
import kgsaf_jdex.utils.conventions.paths as pcc

inputs = [
    DATASET_PATH / pcc.ONTOLOGY,
    DATASET_PATH / pcc.INDIVIDUALS,
    DATASET_PATH / pcc.RDF_TRIPLES,
    DATASET_PATH / pcc.RDF_CLASS_ASSERTIONS
]

output_file = DATASET_PATH / "knowledge_graph.owl"
cmd = ["java", "-Xmx20G", "-jar", str(ROBOT_JAR), "merge"]

for infile in inputs:
    cmd += ["--input", str(infile)]

cmd += ["--output", str(output_file)]

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode != 0:
    raise RuntimeError(f"ROBOT merge failed with return code {result.returncode}")

print(f"Merged knowledge graph saved to {output_file}")

## Machine Learning and Torch Utilities: ID Mapping, TSV Conversion, JSON Conversion

In [ ]:
print(f"\t Converting Dataset {DATASET_PATH.name}")
processor = TSVConverter(DATASET_PATH)
processor.convert()
processor.serialize()

In [ ]:
print(f"\tProcessing Dataset {DATASET_PATH.name}")
processor = OWLConverter(DATASET_PATH)
processor.preprocess(verbose=False)
processor.serialize()

In [ ]:
print(f"Mapping Dataset {DATASET_PATH.name}")
mapper = IDMapper(DATASET_PATH)
mapper.map_to_id()
mapper.serialize()